# 1.Data Cleaning 
 ---

**Assignment:** *Machine Learning 2026 — NOVA School of Business and Economics*  
**Dataset:** Predict students' dropout and academic success   
**Goal:** Identify high-risk students at the end of their first semester using: Socio-economic background, Enrollment data and 1st semester academic performance


 
 *Lavinia Antonino | 71993 Mª Teresa Silva | Leonardo Cantu | Edoardo xxx | Carolina Diogo*

 ---

## Table of Contents

1. [Introduction](#1-introduction)
2. [Data Understanding & Preparation](#2-data-understanding--preparation)
   - [2.1 Load the Dataset & Preview](#21-load-the-dataset--preview)
   - [2.2 Inspect Data Structure](#22-inspect-data-structure)
   - [2.3 Handle Missing Values and Duplicates](#23-handle-missing-values-and-duplicates)
   - [2.4 Variable Types & Grouping](#24-variable-types--grouping)
   - [2.5 Recode Target Variable](#25-recode-target-variable)
3. [Conclusion](#3-conclusion)

---
## 1. Introduction

This notebook is the first part of the project focused on predicting student dropout in higher 
education. The dataset contains academic performance, socioeconomic, and demographic variables collected 
from a higher institution. This notebook covers exclusively the data understanding and preparation phase, 
including data structure inspection, handling of missing values and duplicates, variable type 
identification and grouping, and target variable recoding, ensuring the dataset is clean and 
well-structured for the subsequent EDA, feature selection and modelling stages.

---
## 2. Data Understanding & Preparation


The goal of this section is to ensure the dataset is trustworthy and ready for analysis. We will provide an overview of the data by examining its size, columns, variable types, and unique values. This process will help us identify immediate red flags, determine variables that are irrelevant to the analysis, and highlight those that require adjustments to facilitate a smoother analytical process.

In [46]:
#Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO
import subprocess
import sys
import os
import kagglehub

### 2.1 Load the dataset & preview

The dataste was imported directly trough an API from kaggle. 

Then displayed the first rows to get the initial view of the data.

In [47]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])


# Download latest version
path = kagglehub.dataset_download("thedevastator/higher-education-predictors-of-student-retention")
print("Path to dataset files:", path)


# List files in the downloaded path
print(os.listdir(path))

# Read the dataset
df = pd.read_csv(path + "/dataset.csv")
display(df.head())

Path to dataset files: C:\Users\mtere\.cache\kagglehub\datasets\thedevastator\higher-education-predictors-of-student-retention\versions\2
['dataset.csv']


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


At first look, the dataset appears clean: the variable names are well-structured, and no missing data is observed in the displayed rows. However, a more thorough analysis is needed to confirm this across the entire dataset.

### 2.2 Inspect Data Struture

The `check()` function inspects a dataset for data quality. It returns each column’s data type, number of unique values, non-null values, missing values, and duplicates.

Addicionally, we will use the `applied data.describe().T` function to get a quick statistical summary of all numeric columns, including count, mean, standard deviation, min, max, and quartiles.

Also we will to check the unique values of each of variable to understand it better.

Together, these three steps give a clear overview of the dataset’s structure and quality.

In [48]:
def check(df):
   # Store column statistics
    list=[]

    for col in df.columns:
        columns = df.columns

        # Column data type
        dtype = df[col].dtypes

        # Non-null values
        instances = df[col].count()

        # Unique values
        unique = df[col].nunique()

        # Missing values
        sum_null = df[col].isnull().sum()

        # Duplicate values
        duplicates = df[col].duplicated().sum()
        list.append([dtype,instances,unique,sum_null,duplicates])
    data_check = pd.DataFrame(list,columns=["dtype","instances","unique","sum_null","duplicates"],index=df.columns)
    return data_check

check(df)

,dtype,instances,unique,sum_null,duplicates
Marital status,int64,4424,6,0,4418
Application mode,int64,4424,18,0,4406
Application order,int64,4424,8,0,4416
Course,int64,4424,17,0,4407
Daytime/evening attendance,int64,4424,2,0,4422
Previous qualification,int64,4424,17,0,4407
Nacionality,int64,4424,21,0,4403
Mother's qualification,int64,4424,29,0,4395
Father's qualification,int64,4424,34,0,4390
Mother's occupation,int64,4424,32,0,4392


In [49]:
# The different unique values of each variable
print(df.apply(lambda col: col.unique()))

Marital status                                                                   [1, 2, 4, 3, 5, 6]
Application mode                                  [8, 6, 1, 12, 9, 17, 15, 16, 14, 4, 13, 7, 3, ...
Application order                                                          [5, 1, 2, 4, 3, 6, 9, 0]
Course                                            [2, 11, 5, 15, 3, 17, 12, 10, 14, 16, 6, 8, 13...
Daytime/evening attendance                                                                   [1, 0]
Previous qualification                            [1, 12, 16, 14, 8, 3, 15, 2, 4, 9, 17, 11, 6, ...
Nacionality                                       [1, 15, 3, 14, 12, 18, 5, 11, 8, 17, 4, 9, 13,...
Mother's qualification                            [13, 1, 22, 23, 3, 4, 27, 2, 19, 10, 25, 7, 5,...
Father's qualification                            [10, 3, 27, 28, 1, 14, 5, 4, 24, 2, 29, 9, 7, ...
Mother's occupation                               [6, 4, 10, 8, 5, 2, 16, 1, 7, 3, 12, 9, 20, 28...


The dataset contains **4,424 instances** and **35 variables**, including the target variable. The dataset is composed primarily of **int64** variables, with a smaller set of **float64** variables covering academic grades and macroeconomic indicators, and a single **string** variable representing the target. Overall, the dataset is well-structured and requires minimal cleaning before proceeding to the preparation and modelling stages.

### 2.3 Haddle Missing Values and Duplicates 


**Null or Missing Data**

There are no null values or missing values. Observing the dataset *instances* column is always equal to the number of rows (4424) and *sum_null* column shows 0 for all variables. Thus, no imputation or deletion is needed for our dataset.

**Duplicates Values**

In the table above, we observe that some variables have a lot of duplicates values but they are not row duplicates, just repeated values within columns which is normal for categorical variables.



### 2.4 Variable Types & Grouping


**Variables Type**

Variable types can be determined based on their data type, number of unique values, and the nature of the data. **Continuous variables** are represented as float64 and capture measurements that can take any decimal value, such as curricular unit grades and macroeconomic indicators like unemployment rate, inflation rate, and GDP. **Discrete variables** are represented as int64 and capture countable numeric values, including age at enrollment and all curricular unit counts across both semesters.

**Categorical variables** are represented as int64 or string (object) with no meaningful numeric order, such as marital status, course, nationality, parental qualifications and occupations, and the target variable (Dropout, Graduate, Enrolled). **Ordinal variables** are int64 where values follow a meaningful ranked order, as seen in previous qualification and application order. Finally, **binary variables** are int64 with only two possible values (0 or 1), representing conditions such as displaced, debtor, gender, scholarship holder, and daytime/evening attendance, among others.


This variety will allow for comprehensive analysis across different data types in the EDA.


**Grouping Variables**

Before proceeding with EDA, variables were organized into thematic groups based on their conceptual relationship with the dropout phenomenon. This grouping serves to structure the exploratory analysis and ensure that each dimension of the student profile is considered independently.

**1. Demographic & Personal Background**
Captures intrinsic student characteristics that may influence academic persistence, such as age, gender, marital status, and origin.

**2. Socioeconomic & Family Background**
Reflects the family's educational and professional context, which is known in the literature to be a strong predictor of academic success and dropout risk.

**3. Enrollment & Application Profile**
Describes how and why the student entered the institution, including the course chosen, the time of attendance, and prior academic qualifications.

**4. Financial Status**
Captures the student's financial situation directly, including debt, scholarship support, and tuition payment status, variables strongly associated with dropout in higher education.

**5. 1st Semester Academic Performance**
Reflects early academic behavior and performance, which are among the most immediate indicators of dropout risk.

**6. 2nd Semester Academic Performance**
Captures the evolution of academic performance over time, allowing the model to detect deterioration patterns between semesters.

**7. Macroeconomic Context**
External economic conditions at the time of enrollment that may indirectly influence a student's ability to remain enrolled.

In [50]:
# Grouping variables

demographic = [
    'Marital status',
    'Gender',
    'Age at enrollment',
    'Nacionality',
    'International',
    'Displaced'
]

socioeconomic = [
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    'Educational special needs'
]

enrollment = [
    'Application mode',
    'Application order',
    'Course',
    'Daytime/evening attendance',
    'Previous qualification'
]

financial = [
    'Debtor',
    'Tuition fees up to date',
    'Scholarship holder'
]

academic_1st_sem = [
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (without evaluations)'
]

academic_2nd_sem = [
    'Curricular units 2nd sem (credited)',
    'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 2nd sem (without evaluations)'
]

macroeconomic = [
    'Unemployment rate',
    'Inflation rate',
    'GDP'
]

# Identifying the groups
variable_groups = {
    'Demographic & Personal Background'   : demographic,
    'Socioeconomic & Family Background'   : socioeconomic,
    'Enrollment & Application Profile'    : enrollment,
    'Financial Status'                    : financial,
    '1st Semester Academic Performance'   : academic_1st_sem,
    '2nd Semester Academic Performance'   : academic_2nd_sem,
    'Macroeconomic Context'               : macroeconomic
}

# Verify if all variables were included
all_grouped = [var for group in variable_groups.values() for var in group]
all_features = [col for col in df.columns if col != 'Target']

not_grouped = [var for var in all_features if var not in all_grouped]
print(f"Variables not assigned to any group: {not_grouped}")
print(f"Total grouped: {len(all_grouped)} | Total features: {len(all_features)}")

Variables not assigned to any group: []
Total grouped: 34 | Total features: 34


### 2.5 Recode Target variable

The target variable must be binarized since the current Target has three classes (Dropout, Graduate, Enrolled). Depending on the goal, one approach is to recode it as Dropout vs. Non-Dropout (Graduate + Enrolled), turning it into a binary classification problem.

In [51]:
# Recode Target variable: Dropout = 1, Non-Dropout = 0
df['Target_binary'] = (df['Target'] == 'Dropout').astype(int)

# Verify the recoding
print(df['Target_binary'].value_counts())
print(df[['Target', 'Target_binary']].drop_duplicates())

Target_binary
0    3003
1    1421
Name: count, dtype: int64
      Target  Target_binary
0    Dropout              1
1   Graduate              0
16  Enrolled              0


**Save Cleaned Dataset**

In [52]:
# Save your cleaned dataset into that folder
df.to_csv("../datasets/clean_dataset.csv", index=False)
print("Saved!")

Saved!


---
### 3. Conclusion

The dataset turned out to be remarkably clean, no missing values, no duplicate rows, which made this preparation phase straightforward. With **4,424 students** and **35 variables**, the data covers a wide range of information, from family background and financial situation to academic performance and macroeconomic context. To keep things organized, variables were grouped into **7 thematic groups** that reflect the different dimensions of a student's profile, making it easier to reason about them in the next stages. The target variable was also recoded into a simple binary outcome, so the dataset is now ready for feature selection and modelling.

